# Numeric-feature baseline — UCI Phishing Websites (ID 327)

This research notebook reproduces the methodology of the original project. It is separate from the URL-text model serving `app.py`. All metrics must be regenerated when running this notebook.

## 1. Setup

Install `pip install ucimlrepo pandas matplotlib scikit-learn` in your notebook environment.

In [ ]:
from ucimlrepo import fetch_ucirepo
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score


## 2. Data inspection

Target: `-1` phishing, `1` legitimate. Features are already derived in the source dataset.

In [ ]:
dataset = fetch_ucirepo(id=327)
X = dataset.data.features.copy()
y = dataset.data.targets.iloc[:, 0].copy()
print("Rows / features:", X.shape)
print("Missing values:", X.isna().sum().sum())
print("Class counts:", y.value_counts().to_dict())


## 3. EDA

In [ ]:
ax = y.value_counts().sort_index().plot(kind="bar", title="Class distribution")
ax.set_xlabel("Class (-1 phishing, 1 legitimate)")
ax.set_ylabel("Count")
plt.show()


## 4. Train/test split and model comparison

The 80/20 split matches the original notebook. It is a **row-level** split; these metrics should not be represented as an out-of-domain benchmark.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20, random_state=42)
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
}
for name, estimator in models.items():
    estimator.fit(X_train,y_train)
    pred=estimator.predict(X_test)
    print("\n",name,"accuracy:",round(accuracy_score(y_test,pred),4))
    print(classification_report(y_test,pred,labels=[-1,1],target_names=["Phishing", "Legitimate"]))
    print("Confusion matrix [-1,1]:\n",confusion_matrix(y_test,pred,labels=[-1,1]))


## 5. Feature importances

Impurity-based importance is descriptive, not causal or per-prediction evidence.

In [ ]:
forest = models["Random Forest"]
importance = pd.Series(forest.feature_importances_,index=X.columns).sort_values(ascending=False).head(10)
importance.sort_values().plot.barh(title="Random Forest — top 10 importances")
plt.show()


## Optional deep learning experiment

The original notebook also includes a Keras neural-network comparison. It is intentionally optional here because app deployment does not depend on TensorFlow. Metrics cannot be assumed identical without rerunning.

In [ ]:
# Optional (install tensorflow first):
# from sklearn.preprocessing import StandardScaler
# from tensorflow import keras
# from tensorflow.keras import layers
# scaler=StandardScaler()
# Xtr=scaler.fit_transform(X_train)
# Xte=scaler.transform(X_test)
# model=keras.Sequential([
#     layers.Input(shape=(Xtr.shape[1],)),
#     layers.Dense(32,activation="relu"), layers.Dense(16,activation="relu"),
#     layers.Dense(1,activation="sigmoid"),
# ])
# model.compile(optimizer="adam",loss="binary_crossentropy",metrics=["accuracy"])
# model.fit(Xtr,(y_train==1).astype(int),epochs=20,batch_size=32,verbose=1)
# pred=np.where(model.predict(Xte).ravel()>=.5,1,-1)
# print(classification_report(y_test,pred,labels=[-1,1]))
